# Real-Time ASL to Text Translation Pipeline

## Overview
This notebook bridges the gap between model training and live translation. It covers:
1. Loading and evaluating the trained 26-letter ASL model
2. Real-time inference optimization
3. Model quantization for faster processing
4. Temporal smoothing for consistent predictions
5. Performance profiling (FPS, latency)
6. Integration with the Streamlit app

### Goal
Transform a high-accuracy trained model into a production-ready real-time translator.

## Section 1: Setup & Load Pre-trained Model

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
import cv2
import time
from collections import deque
from datetime import datetime
import json
import pickle
import sys
sys.path.insert(0, '/Users/naveenatik/Documents/projects/sign-language-translator')

from src.pose_detector import PoseDetector
from src.gesture_recognizer import GestureRecognizer
from src.utils import SmoothingFilter, TextBuffer

print("✅ Dependencies loaded")
print(f"📦 TensorFlow version: {tf.__version__}")
print(f"📦 NumPy version: {np.__version__}")
print(f"📦 OpenCV version: {cv2.__version__}")

## Section 2: Model Loading & Evaluation

In [ ]:
# Load the trained model (from optimization notebook)
MODEL_PATH = '/Users/naveenatik/Documents/projects/sign-language-translator/models/asl_model.h5'

try:
    model = load_model(MODEL_PATH)
    print(f"✅ Model loaded: {MODEL_PATH}")
    print(f"\n📊 Model Architecture:")
    print(f"   • Input shape: {model.input_shape}")
    print(f"   • Output shape: {model.output_shape}")
    print(f"   • Parameters: {model.count_params():,}")
    model.summary()
except FileNotFoundError:
    print(f"⚠️  Model not found at {MODEL_PATH}")
    print("📝 First train a model using scripts/train_model.py")
    model = None

## Section 3: Performance Profiling - Baseline

In [ ]:
def profile_inference_performance(model, input_shape=(1, 30, 48), num_samples=100):
    """
    Profile model inference performance on CPU/GPU.
    Measures latency, throughput, and memory usage.
    """
    print(f"🔍 Profiling inference performance...\n")
    
    # Warm up
    dummy_input = np.random.randn(*input_shape).astype(np.float32)
    _ = model.predict(dummy_input, verbose=0)
    
    # Profile
    inference_times = []
    
    for _ in range(num_samples):
        test_input = np.random.randn(*input_shape).astype(np.float32)
        
        start = time.perf_counter()
        _ = model.predict(test_input, verbose=0)
        end = time.perf_counter()
        
        inference_times.append((end - start) * 1000)  # Convert to ms
    
    # Statistics
    inference_times = np.array(inference_times)
    
    results = {
        'mean_latency_ms': np.mean(inference_times),
        'std_latency_ms': np.std(inference_times),
        'min_latency_ms': np.min(inference_times),
        'max_latency_ms': np.max(inference_times),
        'p95_latency_ms': np.percentile(inference_times, 95),
        'p99_latency_ms': np.percentile(inference_times, 99),
        'throughput_fps': 1000 / np.mean(inference_times)
    }
    
    print(f"📊 Inference Performance Metrics:")
    print(f"   • Mean latency: {results['mean_latency_ms']:.2f} ms")
    print(f"   • Std dev: {results['std_latency_ms']:.2f} ms")
    print(f"   • Min latency: {results['min_latency_ms']:.2f} ms")
    print(f"   • Max latency: {results['max_latency_ms']:.2f} ms")
    print(f"   • P95 latency: {results['p95_latency_ms']:.2f} ms")
    print(f"   • P99 latency: {results['p99_latency_ms']:.2f} ms")
    print(f"   • Throughput: {results['throughput_fps']:.1f} FPS")
    
    # Estimate real-world FPS
    frame_capture_time = 1000 / 30  # Assuming 30 FPS camera
    total_frame_time = frame_capture_time + results['mean_latency_ms']
    estimated_fps = 1000 / total_frame_time
    print(f"\n   • Estimated real-world FPS (30 FPS camera): {estimated_fps:.1f} FPS")
    
    return results, inference_times

if model:
    baseline_results, baseline_times = profile_inference_performance(model, num_samples=100)

## Section 4: Model Quantization for Speed

In [ ]:
def quantize_model_to_tflite(keras_model, output_path):
    """
    Convert Keras model to TensorFlow Lite with quantization.
    Reduces model size by ~75% and improves inference speed.
    """
    print(f"🔄 Converting model to TensorFlow Lite...\n")
    
    # Convert to TFLite with dynamic range quantization
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS
    ]
    
    tflite_model = converter.convert()
    
    # Save quantized model
    with open(output_path, 'wb') as f:
        f.write(tflite_model)
    
    print(f"✅ TFLite model saved: {output_path}")
    
    # Compare sizes
    import os
    original_size = os.path.getsize(MODEL_PATH) / (1024 * 1024)  # MB
    quantized_size = os.path.getsize(output_path) / (1024 * 1024)  # MB
    compression_ratio = (1 - quantized_size / original_size) * 100
    
    print(f"\n📊 Model Size Comparison:")
    print(f"   • Original (Keras): {original_size:.2f} MB")
    print(f"   • Quantized (TFLite): {quantized_size:.2f} MB")
    print(f"   • Compression: {compression_ratio:.1f}%")
    
    return tflite_model

if model:
    tflite_output_path = '/Users/naveenatik/Documents/projects/sign-language-translator/models/asl_model.tflite'
    tflite_model = quantize_model_to_tflite(model, tflite_output_path)

## Section 5: Temporal Smoothing Strategies

In [ ]:
class AdvancedTemporalSmoother:
    """
    Advanced smoothing for real-time prediction stability.
    Handles noisy predictions and provides confidence-weighted output.
    """
    def __init__(self, window_size=5, confidence_threshold=0.6):
        self.window_size = window_size
        self.confidence_threshold = confidence_threshold
        self.prediction_history = deque(maxlen=window_size)
        self.confidence_history = deque(maxlen=window_size)
    
    def smooth(self, prediction, confidence):
        """
        Apply weighted temporal smoothing.
        
        Args:
            prediction: Current prediction (label string)
            confidence: Confidence score (0-1)
            
        Returns:
            (smoothed_prediction, is_confident) tuple
        """
        # Only add confident predictions
        if confidence >= self.confidence_threshold:
            self.prediction_history.append(prediction)
            self.confidence_history.append(confidence)
        
        # Need minimum samples
        if len(self.prediction_history) < self.window_size:
            return None, False
        
        # Weighted voting
        from collections import Counter
        predictions = list(self.prediction_history)
        confidences = list(self.confidence_history)
        
        # Weight each prediction by its confidence
        weighted_votes = {}
        for pred, conf in zip(predictions, confidences):
            weighted_votes[pred] = weighted_votes.get(pred, 0) + conf
        
        # Get highest weighted prediction
        best_pred = max(weighted_votes, key=weighted_votes.get)
        avg_confidence = sum(confidences) / len(confidences)
        
        return best_pred, avg_confidence >= self.confidence_threshold
    
    def reset(self):
        """Reset history."""
        self.prediction_history.clear()
        self.confidence_history.clear()

print("✅ AdvancedTemporalSmoother class defined")

# Test smoothing
smoother = AdvancedTemporalSmoother(window_size=5, confidence_threshold=0.6)

# Simulate noisy predictions
test_predictions = [
    ('A', 0.65),
    ('A', 0.72),
    ('B', 0.55),  # Noise
    ('A', 0.68),
    ('A', 0.70),
]

print("\n🧪 Testing smoothing with noisy predictions:")
for i, (pred, conf) in enumerate(test_predictions):
    smooth_pred, is_confident = smoother.smooth(pred, conf)
    print(f"   Frame {i+1}: raw={pred}({conf:.2f}) → smoothed={smooth_pred}({is_confident})")

## Section 6: Continuous Phrase Recognition

In [ ]:
class PhraseRecognizer:
    """
    Converts sequence of letter predictions into phrases.
    Handles timing, spacing, and common patterns.
    """
    def __init__(self, min_frames_between_letters=10):
        self.min_frames_between_letters = min_frames_between_letters
        self.last_letter = None
        self.frames_since_last_letter = 0
        self.phrase = ""
    
    def add_letter(self, letter):
        """
        Add a letter prediction to the phrase.
        
        Args:
            letter: Predicted letter ('A'-'Z', 'space', 'delete')
            
        Returns:
            Updated phrase string
        """
        if letter is None:
            self.frames_since_last_letter += 1
            return self.phrase
        
        # Check if enough frames have passed (avoid double-counting same gesture)
        if letter == self.last_letter and self.frames_since_last_letter < self.min_frames_between_letters:
            return self.phrase
        
        # Add letter to phrase
        if letter == 'space':
            if not self.phrase.endswith(' '):
                self.phrase += ' '
        elif letter == 'delete':
            self.phrase = self.phrase[:-1] if self.phrase else ""
        else:
            self.phrase += letter
        
        self.last_letter = letter
        self.frames_since_last_letter = 0
        
        return self.phrase
    
    def get_phrase(self):
        return self.phrase
    
    def clear(self):
        self.phrase = ""
        self.last_letter = None
        self.frames_since_last_letter = 0

print("✅ PhraseRecognizer class defined")

# Test phrase building
recognizer = PhraseRecognizer(min_frames_between_letters=5)

test_sequence = ['H', 'H', 'E', 'E', 'L', 'L', 'L', 'O', 'O', 'space', 'W', 'O', 'R', 'L', 'D']

print("\n🧪 Testing phrase recognition:")
for letter in test_sequence:
    phrase = recognizer.add_letter(letter)
    print(f"   Letter: {letter} → Phrase: '{phrase}'")

## Section 7: Complete Real-Time Pipeline Simulation

In [ ]:
class RealTimePipeline:
    """
    Complete pipeline for real-time ASL translation.
    Integrates detection, recognition, smoothing, and phrase building.
    """
    def __init__(self, model, confidence_threshold=0.6, smoothing_window=5):
        self.model = model
        self.detector = PoseDetector()
        self.recognizer = GestureRecognizer()
        self.smoother = AdvancedTemporalSmoother(window_size=smoothing_window, confidence_threshold=confidence_threshold)
        self.phrase_builder = PhraseRecognizer()
        
        # Performance tracking
        self.frame_times = deque(maxlen=30)
        self.inference_times = deque(maxlen=30)
    
    def process_frame(self, frame, show_landmarks=True):
        """
        Process a single frame end-to-end.
        
        Returns:
            {
                'frame': processed_frame,
                'prediction': letter,
                'confidence': float,
                'phrase': str,
                'fps': float,
                'inference_ms': float
            }
        """
        frame_start = time.perf_counter()
        
        # 1. Detect hand landmarks
        if show_landmarks:
            annotated_frame, hand_landmarks_list = self.detector.detect(frame)
        else:
            _, hand_landmarks_list = self.detector.detect(frame)
            annotated_frame = frame.copy()
        
        prediction = None
        confidence = 0.0
        inference_time = 0.0
        
        # 2. Recognize gesture (if hands detected)
        if hand_landmarks_list:
            landmarks = hand_landmarks_list[0]
            normalized_landmarks = self.detector.normalize_landmarks(landmarks)
            
            # Inference
            inference_start = time.perf_counter()
            raw_pred = self.recognizer.predict(normalized_landmarks, confidence_threshold=0.3)
            inference_time = (time.perf_counter() - inference_start) * 1000
            
            if raw_pred:
                prediction = raw_pred['label']
                confidence = raw_pred['confidence']
        
        # 3. Apply smoothing
        smoothed_pred, is_confident = self.smoother.smooth(prediction, confidence)
        
        # 4. Build phrase
        phrase = self.phrase_builder.add_letter(smoothed_pred)
        
        # 5. Compute FPS
        frame_time = (time.perf_counter() - frame_start) * 1000
        self.frame_times.append(frame_time)
        self.inference_times.append(inference_time)
        fps = 1000 / np.mean(self.frame_times) if self.frame_times else 0
        
        return {
            'frame': annotated_frame,
            'prediction': smoothed_pred,
            'confidence': confidence,
            'phrase': phrase,
            'fps': fps,
            'inference_ms': inference_time
        }
    
    def reset(self):
        self.smoother.reset()
        self.phrase_builder.clear()

if model:
    pipeline = RealTimePipeline(model, confidence_threshold=0.6, smoothing_window=5)
    print("✅ RealTimePipeline initialized")

## Section 8: Performance Benchmarking Summary

In [ ]:
if model:
    print("\n" + "="*60)
    print("REAL-TIME TRANSLATION READINESS ASSESSMENT")
    print("="*60)
    
    mean_latency = baseline_results['mean_latency_ms']
    throughput = baseline_results['throughput_fps']
    estimated_rtfps = baseline_results.get('estimated_fps', 0)
    
    print(f"\n📊 Performance Metrics:")
    print(f"   • Inference latency: {mean_latency:.2f} ms")
    print(f"   • Max throughput: {throughput:.1f} FPS")
    print(f"   • Real-world estimate (30fps cam): {estimated_rtfps:.1f} FPS")
    
    print(f"\n✅ Optimizations Applied:")
    print(f"   ✓ Temporal smoothing (5-frame window)")
    print(f"   ✓ Continuous phrase recognition")
    print(f"   ✓ Advanced confidence weighting")
    print(f"   ✓ TensorFlow Lite quantization")
    
    print(f"\n🎯 Ready for Deployment:")
    if mean_latency < 50:  # Less than 50ms
        print(f"   ✅ YES - Latency acceptable for real-time translation")
    else:
        print(f"   ⚠️  Consider further optimization")
    
    print(f"\n📁 Generated Artifacts:")
    print(f"   • Keras model: {MODEL_PATH}")
    print(f"   • TFLite model: models/asl_model.tflite")
    print(f"   • Pipeline ready for Streamlit app")

## Summary & Next Steps

### ✅ What This Notebook Provides

1. **Performance Profiling**
   - Baseline latency measurement
   - Real-world FPS estimation
   - Throughput analysis

2. **Model Optimization**
   - TensorFlow Lite quantization
   - 75% smaller model size
   - Faster inference on mobile/edge

3. **Real-Time Smoothing**
   - Temporal prediction smoothing
   - Confidence-weighted voting
   - Noise reduction

4. **Phrase Recognition**
   - Continuous letter-to-phrase conversion
   - Gesture timing handling
   - Delete/space support

5. **Complete Pipeline**
   - End-to-end processing
   - FPS tracking
   - Ready for Streamlit integration

### 🚀 Next Steps

1. **Run Streamlit App**
   ```bash
   streamlit run app.py
   ```

2. **Live Testing**
   - Test with real webcam
   - Measure actual FPS
   - Verify accuracy

3. **Further Improvements**
   - Collect more diverse training data
   - Fine-tune temporal smoothing
   - Add continuous word recognition
   - Implement multi-hand coordination

### 📈 Expected Results

- **Model Accuracy**: 85-90% on 26 letters
- **Inference Speed**: <50ms per frame
- **Real-time FPS**: 15-30 FPS (depends on hardware)
- **User Experience**: Smooth, responsive translation